# Single-voxel drift, Roth & Merriam (2023) procedure — V1–hV4

Default: **SP 4 orientations × 4 SF (18 predictors), raw betas**. Other normalizations, TS and 6-level SP models can be switched on in the configuration cell.

Cross-session generalization of session-wise encoding models, analysed exactly as in Roth & Merriam (2023):

- **Voxels:** all ROI voxels with NSD pRF R² > 0 (no noise-ceiling filter, no exclusion of negative values).
- **Aggregation:** for each subject and session pair, the **median** across voxels → one 30 × 30 matrix per subject (rows = training session, columns = test session).
- **Drift statistic:** Pearson correlation between goodness-of-fit and session lag over all off-diagonal entries, per subject, averaged across subjects.
- **Significance:** 1000 permutations of session order (same permutation for all subjects); one-sided, drift = negative correlation.
- **Metrics:** cross-validated R² (sensitive to baseline and gain) and Pearson r (insensitive to both).
- **Normalizations (SP):** raw, `zeroMean` (session mean removed), `equalStd` (session variance equalized, mean kept).

**Matched voxel-count control:** the per-voxel values do not depend on ROI size, but the median across voxels is noisier for small ROIs, and noise in the matrix attenuates the lag correlation. The control subsamples every ROI of a subject to that subject's smallest ROI (one seeded random draw per subject × ROI, same voxels for all models; same seed scheme as SP notebook 6) and repeats the analysis.

Inputs (read-only): TS `FAST_CROSS_SESSION/*.h5py`, SP `regressPrfSplit_session_v*_sub*{norm}.pkl`, NSD `prf-visualrois` and pRF `R2`. Models whose files are missing are skipped.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ==== Configuration ====
import glob, os, pickle, h5py, numpy as np, nibabel as nib, matplotlib.pyplot as plt

FITS   = "/content/drive/MyDrive/V2_drift/texturemodel/model_fits/"
SPDIR  = "/content/drive/MyDrive/V1_Drift/repDrift_expand/"           # SP 4 ori x 4 SF (18 predictors)
SPDIR6 = "/content/drive/MyDrive/V1_Drift/repDrift_expand_6levels/"   # SP 4 ori x 6 SF (26 predictors)
NSD    = "/content/drive/MyDrive/V1_Drift/NSD/"
OUT    = "/content/drive/MyDrive/V1_Drift/plots/single_voxel_roth_style/"

CODES   = {"V1": [1, 2], "V2": [3, 4], "V3": [5, 6], "hV4": [7]}
SP_VREG = {"V1": 1, "V2": 2, "V3": 3, "hV4": 4}

# TS session-wise runs (FAST_CROSS_SESSION): name -> file tag. Missing files are skipped.
# Default: SP 4 ori x 4 SF only. Uncomment lines to add models.
TS_CFG = {
    # "TS drift (54), raw":     "pcaHL_{roi}_pca95",
    # "TS LL only (49), raw":   "pcaHL_{roi}_pca0",
    # "TS k1 (58), raw":        "pcaHL_HendersonGroups_{roi}_k1",
}
# SP session-wise runs: name -> (folder, normalization suffix). Missing files are skipped.
SP_SOURCES = {
    "SP (18), raw":            (SPDIR,  ""),
    # "SP (18), zeroMean":       (SPDIR,  "_zeroMean"),
    # "SP (18), equalStd":       (SPDIR,  "_equalStd"),
    # "SP 6lev (26), raw":       (SPDIR6, ""),
    # "SP 6lev (26), zeroMean":  (SPDIR6, "_zeroMean"),
    # "SP 6lev (26), equalStd":  (SPDIR6, "_equalStd"),
}
NSESS, NPERM = 30, 1000
MATCH_SEED   = 0          # matched voxel count: one random draw per subject x ROI (same seed scheme as SP notebook 6)
SUBJECTS     = range(1, 9)


In [ ]:
# ==== Functions ====
def ts_matrices(s, roi, tag, good):
    fns = glob.glob(FITS + f"S{s:02d}/FAST_CROSS_SESSION/S{s:02d}_{roi}_texture_pyramid_OLS_4ori_4sf_"
                    f"{tag.format(roi=roi)}_use_nsd_grid5_pRFs_*cross_session.h5py")
    if not fns: return None
    with h5py.File(fns[0], "r") as f:
        keep = good[np.asarray(f["voxel_index"]).ravel()]
        out = {k: np.nanmedian(np.asarray(f[k][:NSESS, :NSESS, :, 0])[:, :, keep], axis=2)
               for k in ["val_r2", "val_cc"]}                       # [train, test]
    out["nvox"] = int(keep.sum()); return out

def sp_matrices(s, roi, folder, z, good):
    fn = folder + f"regressPrfSplit_session_v{SP_VREG[roi]}_sub{s}{z}.pkl"
    if not os.path.exists(fn): return None
    with open(fn, "rb") as f: D = pickle.load(f)
    nsd, out = D["nsd"], {}
    vi = np.concatenate([np.asarray(nsd["roiInd"][rn]).ravel() for rn in D["rois"]]); keep = good[vi]
    for key, name in [("r2oriSplit", "val_r2"), ("pearsonRori", "val_cc")]:
        M = np.concatenate([np.asarray(nsd[key][rn]) for rn in D["rois"]], axis=1)   # [data sess, vox, model sess]
        out[name] = np.nanmedian(M[:NSESS, keep, :NSESS], axis=1).T                  # -> [train(model), test(data)]
    out["nvox"] = int(keep.sum()); del D, nsd; return out

def voxel_sets(s, match):
    """Roth & Merriam: all ROI voxels with NSD pRF R2 > 0. match=True: subsample every ROI to the
    subject's smallest ROI (one seeded draw per subject x ROI)."""
    ld = lambda f: nib.load(NSD + f"subj{s:02d}/func1pt8mm/{f}").get_fdata().ravel()
    vis, pr2 = ld("roi/prf-visualrois.nii.gz"), ld("R2.nii.gz")
    goods = {roi: np.isin(vis, codes) & (pr2 > 0) for roi, codes in CODES.items()}
    counts = {roi: int(g.sum()) for roi, g in goods.items()}
    if match:
        n_min = min(counts.values())
        for ri, roi in enumerate(CODES):
            idx = np.random.default_rng([MATCH_SEED, s, ri]).choice(np.flatnonzero(goods[roi]), n_min, replace=False)
            goods[roi] = np.zeros_like(goods[roi]); goods[roi][idx] = True
    return goods, counts

def collect(match):
    res, nvox = {}, {}
    for s in SUBJECTS:
        goods, counts = voxel_sets(s, match)
        print(f"S{s}: voxels with pRF R2 > 0 per ROI {counts}" + (f" -> matched n = {min(counts.values())}" if match else ""))
        for roi in CODES:
            mats = {m: ts_matrices(s, roi, t, goods[roi]) for m, t in TS_CFG.items()}
            mats.update({m: sp_matrices(s, roi, d, z, goods[roi]) for m, (d, z) in SP_SOURCES.items()})
            for m, d in mats.items():
                if d is None: continue
                nvox.setdefault((m, roi), []).append(d["nvox"])
                for k in ["val_r2", "val_cc"]: res.setdefault((m, roi, k), []).append(d[k])
    return res, nvox

lag = np.abs(np.subtract.outer(np.arange(NSESS), np.arange(NSESS))); off = lag > 0
def drift_r(G): return np.corrcoef(G[off], lag[off])[0, 1]
_rng = np.random.default_rng(0); perms = [_rng.permutation(NSESS) for _ in range(NPERM)]

def stats(Gs):
    """Roth & Merriam: drift = Pearson r(goodness-of-fit, lag) over off-diagonal entries, per subject,
    averaged over subjects; p from 1000 session-order permutations (same permutation for all subjects)."""
    obs = np.mean([drift_r(G) for G in Gs])
    null = np.array([np.mean([drift_r(G[np.ix_(p, p)]) for G in Gs]) for p in perms])
    curve = np.mean([[G[lag == d].mean() for d in range(1, NSESS)] for G in Gs], 0)
    return obs, np.mean(null <= obs), curve

def table(res, nvox, title):
    models = list(TS_CFG) + list(SP_SOURCES); rows = {}
    print(f"\n==== {title} ====")
    print(f"{'model':24s} {'ROI':4s} {'metric':6s} {'n':>2s} {'med vox':>7s} {'lag1':>8s} {'lag29':>8s} {'drift r':>8s} {'p':>7s}")
    for roi in CODES:
        for m in models:
            for k, kname in [("val_r2", "cvR2"), ("val_cc", "r")]:
                Gs = res.get((m, roi, k))
                if not Gs: continue
                obs, p, curve = stats(Gs); rows[(m, roi, k)] = (obs, p, curve)
                print(f"{m:24s} {roi:4s} {kname:6s} {len(Gs):2d} {int(np.median(nvox[(m, roi)])):7d} "
                      f"{curve[0]:8.3f} {curve[-1]:8.3f} {obs:+8.3f} {p:7.3f}")
        print()
    return rows

COLS = {"TS drift (54), raw": "#222222", "TS LL only (49), raw": "#888888", "TS k1 (58), raw": "#8c564b",
        "SP (18), raw": "#4a90c4", "SP (18), zeroMean": "#2ca02c", "SP (18), equalStd": "#d62728",
        "SP 6lev (26), raw": "#9467bd", "SP 6lev (26), zeroMean": "#8fd18f", "SP 6lev (26), equalStd": "#f19c9c"}

def plot(rows, title, fname):
    fig, axs = plt.subplots(2, 4, figsize=(17, 6.5), sharex=True)
    for j, roi in enumerate(CODES):
        for i, (k, kname) in enumerate([("val_r2", "cvR2 (median voxel)"), ("val_cc", "r (median voxel)")]):
            ax = axs[i, j]
            for m in list(TS_CFG) + list(SP_SOURCES):
                if (m, roi, k) in rows:
                    ax.plot(range(1, NSESS), rows[(m, roi, k)][2], color=COLS.get(m, "k"), lw=2, label=m)
            ax.axhline(0, color="k", lw=0.5); ax.set_title(roi if i == 0 else "")
            ax.set_ylabel(kname if j == 0 else "")
            for side in ["top", "right"]: ax.spines[side].set_visible(False)
        axs[1, j].set_xlabel("session lag")
    axs[0, 0].legend(frameon=False, fontsize=7); fig.suptitle(title); plt.tight_layout()
    os.makedirs(OUT, exist_ok=True); plt.savefig(OUT + fname, dpi=200); plt.show()
    print("saved:", OUT + fname)


## Run
All voxels first, then the matched voxel count. Each run prints the per-subject voxel counts, a table per ROI × model × metric, and the lag curves. Several minutes per run (the permutations and loading the SP pickles).

In [ ]:
# ==== Run: all voxels, then matched voxel count ====
res_all, n_all = collect(match=False)
rows_all = table(res_all, n_all, "ALL voxels (pRF R2 > 0)")
plot(rows_all, "Single-voxel drift, all voxels", "single_voxel_all.png")

res_m, n_m = collect(match=True)
rows_m = table(res_m, n_m, "MATCHED voxel count per subject")
plot(rows_m, "Single-voxel drift, matched voxel count", "single_voxel_matched.png")


## All voxels vs. matched voxel count
If the ROI pattern of the drift statistic is unchanged after matching, the ROI differences are not driven by ROI size.

In [ ]:
# ==== Drift statistic: all voxels vs matched voxel count ====
print(f"{'model':24s} {'metric':6s} " + "  ".join(f"{r:>17s}" for r in CODES))
print(f"{'':31s}" + "  ".join(f"{'all -> matched':>17s}" for _ in CODES))
for m in list(TS_CFG) + list(SP_SOURCES):
    for k, kname in [("val_r2", "cvR2"), ("val_cc", "r")]:
        if not all((m, roi, k) in rows_all and (m, roi, k) in rows_m for roi in CODES): continue
        cells = []
        for roi in CODES:
            a, pa, _ = rows_all[(m, roi, k)]; b, pb, _ = rows_m[(m, roi, k)]
            sig = lambda p: "*" if p < 0.05 else " "
            cells.append(f"{a:+.3f}{sig(pa)} -> {b:+.3f}{sig(pb)}")
        print(f"{m:24s} {kname:6s} " + "  ".join(f"{c:>17s}" for c in cells))
print("\n* = permutation p < 0.05 (one-sided, drift = negative correlation with lag)")
